# Lesson 12: Synthetic Control Methods

## Opening Story: German Reunification

In 1990, West Germany reunified with East Germany. This created a natural experiment: how did reunification affect the German economy?

Abadie, Diamond, and Hainmueller (2015) used synthetic control methods to construct a "synthetic West Germany" from a weighted combination of other countries that didn't experience reunification. By comparing actual West Germany to its synthetic counterpart, they estimated the causal effect of reunification.

---

## Learning Objectives

By the end of this lesson, you should be able to:

1. Explain the synthetic control method
2. Construct synthetic controls using optimization
3. Conduct inference with placebo tests
4. Assess the quality of synthetic controls
5. Apply the method to policy evaluation

---

## 12.1 The Synthetic Control Method

### The Idea

Construct a counterfactual for the treated unit as a weighted combination of untreated units:

$$Y_{it}(0) = \sum_{j=2}^{J+1} w_j Y_{jt}$$

where the weights $w_j$ are chosen to match pre-treatment characteristics and outcomes.

### The Algorithm

1. Choose donor pool (untreated units)
2. Optimize weights to minimize pre-treatment fit
3. Compare post-treatment outcomes

---

## 12.2 Implementation

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

# Simulated example: 1 treated unit, 10 donor units
n_years = 20
treatment_year = 15
true_effect = -2.0

# Pre-treatment outcomes for treated unit
pre_treatment = np.cumsum(np.random.normal(0.1, 0.5, n_years))

# Donor units (similar trajectories)
donors = np.array([
    pre_treatment + np.random.normal(0, 0.3, n_years) + np.random.normal(0, 0.1) * np.arange(n_years)
    for _ in range(10)
])

# Post-treatment with effect
post_treatment = pre_treatment[-1] + np.cumsum(np.random.normal(0.1, 0.5, n_years - treatment_year))
post_treatment += true_effect  # Add treatment effect

# Combine
y_treated = np.concatenate([pre_treatment, post_treatment])
y_donors = np.column_stack([
    np.column_stack([d[:treatment_year] for d in donors]),
    np.column_stack([d[treatment_year:] for d in donors])
])

# Synthetic control weights (simplified optimization)
from scipy.optimize import minimize

def synthetic_control_weights(y_treated_pre, y_donors_pre):
    """
    Find weights that minimize pre-treatment discrepancy.
    """
    n_donors = y_donors_pre.shape[1]
    
    def objective(w):
        synthetic = y_donors_pre @ w
        return np.sum((y_treated_pre - synthetic) ** 2)
    
    # Constraints: weights sum to 1, non-negative
    constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
    bounds = [(0, 1)] * n_donors
    
    # Initial weights (equal)
    w0 = np.ones(n_donors) / n_donors
    
    result = minimize(objective, w0, bounds=bounds, constraints=constraints)
    return result.x

# Get weights
w = synthetic_control_weights(y_treated[:treatment_year], y_donors[:treatment_year])

# Construct synthetic control
y_synthetic = y_donors @ w

# Estimate treatment effect
effect = y_treated[treatment_year:] - y_synthetic[treatment_year:]
print(f"Estimated treatment effect (post-treatment mean): {effect.mean():.3f}")
print(f"True treatment effect: {true_effect}")

---

## 12.3 Placebo Tests

In [ ]:
# Permutation inference: apply method to each donor unit
effects_placebo = []
for i in range(10):
    # Use donor i as "treated"
    other_donors = np.delete(y_donors, i, axis=1)
    w_placebo = synthetic_control_weights(
        y_donors[:treatment_year, i],
        other_donors[:treatment_year]
    )
    y_synth_placebo = other_donors @ w_placebo
    effect_placebo = y_donors[treatment_year:, i] - y_synth_placebo[treatment_year:]
    effects_placebo.append(effect_placebo.mean())

# Calculate p-value
p_value = np.mean([abs(e) >= abs(effect.mean()) for e in effects_placebo])
print(f"Placebo p-value: {p_value:.3f}")

---

## 12.4 Common Mistakes

1. **Poor pre-treatment fit**: Check RMSPE
2. **Spillovers**: Ensure donor units aren't affected
3. **Overfitting**: Use cross-validation for weight selection
4. **Inference**: Always conduct placebo tests

---

## 12.5 Knowledge Check

### Multiple Choice

1. **Synthetic control constructs:**
   - A) A randomized experiment
   - B) A counterfactual from weighted donors
   - C) A difference-in-differences estimate
   - D) An instrumental variable

2. **Weights are chosen to:**
   A) Maximize treatment effect
   B) Minimize pre-treatment discrepancy
   C) Be equal
   D) Be binary

3. **Placebo tests assess:**
   A) Statistical significance
   B) Practical significance
   C) Both A and B
   D) Neither

4. **The donor pool should:**
   A) Include the treated unit
   B) Not be affected by treatment
   C) Be as large as possible
   D) Be similar to the treated unit

5. **Synthetic control is useful when:**
   A) There's one treated unit
   B) There's randomization
   C) There's a clear cutoff
   D) There are many treated units

### Short Answer

6. **Explain how synthetic control differs from matching.**

7. **Why are placebo tests important?**

8. **How do you assess the quality of a synthetic control?**

9. **What is the role of pre-treatment fit?**

10. **Give an example where synthetic control would be appropriate.**

---

## 12.6 Summary

1. **Synthetic control** constructs counterfactuals from weighted donors
2. **Pre-treatment fit** is crucial for validity
3. **Placebo tests** provide inference
4. **Single-unit studies** can be credible with this method
5. **Donor pool selection** affects results

---

## 12.7 Further Reading

- Abadie, A., Diamond, A., & Hainmueller, J. (2015). "Comparative Politics and the Synthetic Control Method." *American Journal of Political Science*.
- Abadie, A. (2021). "Using Synthetic Control: Feasibility, Data Requirements, and Methodological Aspects." *Journal of Economic Literature*.